CCSD-seeded LUCJ + variational optimization + SQD (H2O CAS(4e,4o))
1. same Pauli Hamiltonian + CCSD amplitudes (t1, t2)
2. seed with `UCJOpSpinBalanced.from_t_amplitudes(...)`
3. handle ordering: ffsim is spin-BLOCKED, the Pauli file is INTERLEAVED -> remap the Hamiltonian
4. GENUINE variational minimization of <H> over the LUCJ parameters (optimize=True only fits the factorization)
5. local pairs aa=[(0,1),(1,2),(2,3)], ab=[(0,0),(1,1),(2,2),(3,3)]; test n_reps = 3 and 4
6. noiseless: variational error and SQD@20 determinants
7. transpile for the IBM backend, record 2Q depth + physical layout
8. 10,000 QPU shots, SQD sweep 5/10/15/20/25/30/36, valid-shot fraction, mean +/- spread over seeds

In [1]:
%pip install -q qiskit qiskit-addon-sqd qiskit-ibm-runtime ffsim scipy matplotlib

In [2]:
import os, sys, json
import numpy as np
import ffsim
import scipy.optimize as so
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
import qiskit_addon_sqd.configuration_recovery as sqd_cr
import qiskit_addon_sqd.counts as sqd_counts
import qiskit_addon_sqd.qubit as sqd_qubit
import qiskit_addon_sqd.subsampling as sqd_sub

In [3]:
if not os.path.exists("SQD"):
    !git clone https://github.com/KhushiPandey1805/SQD.git SQD
    print("Cloned fresh.")
else:
    os.chdir("SQD")
    !git pull
    os.chdir("/content")
    print("Already present, pulled latest changes.")

!ls /content/SQD/code

REPO = "/content/SQD"                      # cloned repo (git clone https://github.com/KhushiPandey1805/SQD.git)
FILEPATH = f"{REPO}/hamiltonian/H2O_CAS4e4o_JW_Pauli_Hamiltonian_HERMITIAN.txt"
AMPS_PATH = f"{REPO}/hamiltonian/H2O_CAS4e4o_CCSD_t_amplitudes.npz"   # put the .npz next to the Hamiltonian
RESULTS_DIR = f"{REPO}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.insert(0, f"{REPO}/code")
from hamiltonianCode import interleaved_to_blocked_layout, parse_hamiltonian_file, parse_metadata

Cloning into 'SQD'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 154 (delta 69), reused 117 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 989.09 KiB | 5.32 MiB/s, done.
Resolving deltas: 100% (69/69), done.
Cloned fresh.
hamiltonianCode.ipynb  N2ex.ipynb	    validation.ipynb
hamiltonianCode.py     simVShardware.ipynb


In [4]:
N_REPS_LIST = (3, 4)
SHOTS = 10_000
SWEEP_SIZES = [5, 10, 15, 20, 25, 30, 36]
N_SEEDS = 10
BACKEND_NAME = "ibm_phoenix"        # e.g. "ibm_<name>" for the Phoenix-class device; None -> least busy
RUN_HARDWARE = True       # set True only after the noiseless numbers look right

1-3. Load Hamiltonian (-> blocked ordering) and CCSD amplitudes

In [5]:
def load_system():
    ham = parse_hamiltonian_file(FILEPATH)
    meta = parse_metadata(FILEPATH)
    norb = meta["n_spatial_orbitals"]
    assert ham.num_qubits == 2 * norb
    if meta["qubit_ordering"] == "interleaved":
        ham = ham.apply_layout(interleaved_to_blocked_layout(norb), num_qubits=2 * norb).simplify()
    elif meta["qubit_ordering"] != "blocked":
        raise ValueError("Unknown qubit ordering in header")
    meta["nelec"] = (meta["n_alpha"], meta["n_beta"])
    meta["norb"] = norb
    amps = np.load(AMPS_PATH)
    return ham, meta, amps["t1"], amps["t2"]

In [6]:
def local_pairs(norb):
    pairs_aa = [(p, p + 1) for p in range(norb - 1)]
    pairs_ab = [(p, p) for p in range(norb)]
    return pairs_aa, pairs_ab

4. Variational minimization
The energy is evaluated in ffsim's (n_alpha, n_beta) subspace (36 states) for speed; the same parameters are then re-checked on the actual qubit circuit (`verify_on_circuit`).

In [7]:
class Problem:
    def __init__(self, ham, meta):
        self.ham, self.meta = ham, meta
        norb, (na, nb) = meta["norb"], meta["nelec"]
        self.norb, self.nelec = norb, (na, nb)
        import math
        a_str = ffsim.addresses_to_strings(range(math.comb(norb, na)), norb, na)
        b_str = ffsim.addresses_to_strings(range(math.comb(norb, nb)), norb, nb)
        # blocked qubit order: alpha orbital k = qubit k, beta orbital k = qubit norb+k
        self.idx = np.array([int(a) | (int(b) << norb) for a in a_str for b in b_str])
        Hm = ham.to_matrix(sparse=True)
        self.Hsub = Hm[self.idx][:, self.idx].toarray()
        self.hf = ffsim.hartree_fock_state(norb, self.nelec)
        self.Eref = meta["reference_energy"]
 
    def op(self, x, n_reps, pairs):
        return ffsim.UCJOpSpinBalanced.from_parameters(
            x, norb=self.norb, n_reps=n_reps, interaction_pairs=pairs,
            with_final_orbital_rotation=True)   # needed because the t1 seed adds a final orbital rotation
 
    def state(self, x, n_reps, pairs):
        return ffsim.apply_unitary(self.hf, self.op(x, n_reps, pairs), norb=self.norb, nelec=self.nelec)
 
    def energy(self, x, n_reps, pairs):
        v = self.state(x, n_reps, pairs)
        return float((v.conj() @ self.Hsub @ v).real)

In [8]:
def seed_parameters(t1, t2, n_reps, pairs):
    op0 = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
        t2=t2, t1=t1, n_reps=n_reps, interaction_pairs=pairs,
        optimize=True, options={"maxiter": 1000})   # factorization fit ONLY -- not the energy
    return op0.to_parameters(interaction_pairs=pairs)

In [9]:
def variational_minimize(prob, x0, n_reps, pairs, n_restarts=3, noise=0.02, seed=0):
    """Genuine minimization of <H> over all LUCJ parameters, from the CCSD seed
    plus a few small random perturbations; best result kept."""
    starts = [("seed", x0)] + [
        (f"seed+noise{s}", x0 + np.random.default_rng(seed + s).normal(0, noise, len(x0)))
        for s in range(n_restarts)]
    best = None
    for tag, start in starts:
        r = so.minimize(prob.energy, start, args=(n_reps, pairs), method="L-BFGS-B",
                        options={"maxiter": 3000, "maxfun": 10**6})
        print(f"   {tag}: error {(r.fun - prob.Eref) * 1e3:.4f} mHa (nit={r.nit})")
        if best is None or r.fun < best.fun:
            best = r
    return best

Circuit + ordering / particle-number checks

In [10]:
def build_circuit(prob, x, n_reps, pairs):
    q = QuantumRegister(2 * prob.norb, "q")
    qc = QuantumCircuit(q)
    qc.append(ffsim.qiskit.PrepareHartreeFockJW(prob.norb, prob.nelec), q)
    qc.append(ffsim.qiskit.UCJOpSpinBalancedJW(prob.op(x, n_reps, pairs)), q)
    return qc

In [11]:
def verify_on_circuit(prob, x, n_reps, pairs):
    """Energy from the real qubit circuit must equal the ffsim-subspace energy,
    and all probability must sit in the (n_alpha, n_beta) sector."""
    qc = build_circuit(prob, x, n_reps, pairs)
    sv = Statevector(qc.decompose(reps=3))
    e_circ = float(np.real(sv.expectation_value(prob.ham)))
    probs = sv.probabilities()
    in_sector = float(probs[prob.idx].sum())
    n = prob.norb
    ks = np.arange(2 ** (2 * n))
    na = np.array([bin(k & (2**n - 1)).count("1") for k in ks])
    nb = np.array([bin(k >> n).count("1") for k in ks])
    exp_na, exp_nb = (probs * na).sum(), (probs * nb).sum()
    print(f"   circuit energy error {(e_circ - prob.Eref) * 1e3:.4f} mHa | "
          f"ffsim-subspace error {(prob.energy(x, n_reps, pairs) - prob.Eref) * 1e3:.4f} mHa | "
          f"weight in (Na,Nb) sector = {in_sector:.8f} | <Na>={exp_na:.4f} <Nb>={exp_nb:.4f}")
    assert abs(e_circ - prob.energy(x, n_reps, pairs)) < 1e-6, "ORDERING CHECK FAILED"
    assert in_sector > 1 - 1e-8, "PARTICLE-NUMBER CHECK FAILED"
    return qc

SQD post-processing (unchanged from the previous workflow)

In [12]:
def report_validity_fractions(bitstring_matrix, probs, meta, label):
    norb, na, nb = meta["norb"], meta["n_alpha"], meta["n_beta"]
    mask = ((bitstring_matrix[:, :norb].sum(axis=1) == nb) & (bitstring_matrix[:, norb:].sum(axis=1) == na))
    f_shots = float(probs[mask].sum()); f_unique = float(mask.sum() / len(mask))
    print(f"[{label}] valid-shot fraction {f_shots:.4f} | valid distinct bitstrings {f_unique:.4f} "
          f"| unique sampled {len(mask)}")
    return f_shots, f_unique

In [13]:
def run_sqd_postprocessing(hamiltonian, meta, bitstring_matrix, probs, n_iterations=5,
                           samples_per_batch=20, num_batches=3, seed=0, verbose=True, label="run"):
    norb, na, nb = meta["norb"], meta["n_alpha"], meta["n_beta"]
    avg_a = np.array([1.0] * na + [0.0] * (norb - na)); avg_b = np.array([1.0] * nb + [0.0] * (norb - nb))
    hist = []
    for it in range(n_iterations):
        bs_r, p_r = sqd_cr.recover_configurations(bitstring_matrix, probs, (avg_a, avg_b),
                                                   num_elec_a=na, num_elec_b=nb, rand_seed=seed + it)
        bs_p, p_p = sqd_sub.postselect_by_hamming_right_and_left(bs_r, p_r, hamming_right=na, hamming_left=nb)
        batches = sqd_sub.subsample(bs_p, p_p, samples_per_batch=min(samples_per_batch, bs_p.shape[0]),
                                    num_batches=num_batches, rand_seed=seed + it)
        best = None
        for b in batches:
            s = sqd_qubit.sort_and_remove_duplicates(b)
            ev, evec = sqd_qubit.solve_qubit(s, hamiltonian, k=1)
            if best is None or ev[0] < best[0]:
                best = (ev[0], evec[:, 0], s)
        w = np.abs(best[1]) ** 2
        occ = ((w[:, None] * best[2]).sum(axis=0))[::-1]
        avg_a, avg_b = occ[:norb], occ[norb:]
        hist.append(best[0])
        if verbose:
            print(f"[{label}] iter {it}: error {(best[0] - meta['reference_energy']) * 1e3:.4f} mHa")
    return hist

In [14]:
def sqd_sweep(hamiltonian, meta, bs, probs, sizes=SWEEP_SIZES, n_seeds=N_SEEDS, label="run"):
    """Final SQD error (mHa) vs samples_per_batch: mean +/- std over subsampling seeds."""
    out = {}
    for spb in sizes:
        errs = [(run_sqd_postprocessing(hamiltonian, meta, bs, probs, n_iterations=3,
                                        samples_per_batch=spb, num_batches=3, seed=sd,
                                        verbose=False, label=label)[-1] - meta["reference_energy"]) * 1e3
                for sd in range(n_seeds)]
        out[spb] = (float(np.mean(errs)), float(np.std(errs)))
        print(f"[{label}] samples_per_batch={spb:>2}: error {out[spb][0]:.4f} +/- {out[spb][1]:.4f} mHa "
              f"(n_seeds={n_seeds})")
    return out

6. Noiseless benchmark for n_reps = 3, 4

In [16]:
def noiseless_stage():
    ham, meta, t1, t2 = load_system()
    pairs = local_pairs(meta["norb"])
    prob = Problem(ham, meta)
    print(f"Exact sector energy error: {(np.linalg.eigvalsh(prob.Hsub)[0] - prob.Eref) * 1e3:.2e} mHa "
          f"| HF error: {(prob.energy(np.zeros(0), 0, pairs) if False else (prob.hf.conj() @ prob.Hsub @ prob.hf).real - prob.Eref) * 1e3:.3f} mHa")
    results = {}
    for n_reps in N_REPS_LIST:
        print(f"\n=== local pairs, n_reps={n_reps} ===")
        x0 = seed_parameters(t1, t2, n_reps, pairs)
        print(f"   n_params={len(x0)}; raw CCSD-seeded error {(prob.energy(x0, n_reps, pairs) - prob.Eref) * 1e3:.3f} mHa")
        best = variational_minimize(prob, x0, n_reps, pairs)
        print(f"   VARIATIONAL error after minimization: {(best.fun - prob.Eref) * 1e3:.4f} mHa")
        qc = verify_on_circuit(prob, best.x, n_reps, pairs)
        np.save(f"{RESULTS_DIR}/lucj_local_r{n_reps}_params.npy", best.x)
        # sample the ideal state -> SQD
        circ = qc.copy(); circ.measure_all()
        sampler = StatevectorSampler(seed=np.random.default_rng(0))
        job = sampler.run([circ.decompose(reps=3)], shots=SHOTS)
        bs, pr = sqd_counts.bit_array_to_arrays(job.result()[0].data.meas)
        report_validity_fractions(bs, pr, meta, f"sim r{n_reps}")
        sweep = sqd_sweep(ham, meta, bs, pr, label=f"sim r{n_reps}")
        results[n_reps] = dict(x=best.x, var_err_mha=(best.fun - prob.Eref) * 1e3, sweep=sweep, circuit=qc)
    json.dump({r: {"var_err_mha": v["var_err_mha"], "sqd_sweep": {str(k): s for k, s in v["sweep"].items()}}
               for r, v in results.items()}, open(f"{RESULTS_DIR}/noiseless_results.json", "w"), indent=2)
    return ham, meta, prob, pairs, results

7-8. Transpile for the IBM backend and run on the QPU

In [15]:
def transpile_for_backend(qc, backend, norb, pairs):
    """Transpile two ways and keep the shallower 2Q depth:
    (a) ffsim's heavy-hex LUCJ pass manager (note: it may drop ab-pairs it cannot place on heavy-hex
        for the *layout* choice; the circuit itself is unchanged), (b) preset optimization_level=3.
    The chosen circuit, its 2Q depth and its physical layout are returned/recorded."""
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    circ = qc.copy(); circ.measure_all()
    cands = {}
    try:
        pm, _ = ffsim.qiskit.generate_lucj_pass_manager(
            backend=backend, norb=norb, connectivity="heavy-hex",
            interaction_pairs=pairs, optimization_level=3)
        cands["ffsim LUCJ pass manager"] = pm.run(circ)
    except Exception as e:
        print("ffsim pass manager failed:", repr(e)[:150])
    cands["preset opt-level 3"] = generate_preset_pass_manager(
        backend=backend, optimization_level=3, seed_transpiler=0).run(circ.decompose(reps=3))
    depth = lambda c: c.depth(lambda i: i.operation.num_qubits == 2)
    for k, c in cands.items():
        print(f"   {k}: 2Q depth={depth(c)}, total depth={c.depth()}")
    how = min(cands, key=lambda k: depth(cands[k]))
    isa = cands[how]; d2 = depth(isa)
    try:
        layout = [int(q) for q in isa.layout.final_index_layout()]
    except Exception:
        layout = None
    print(f"USING {how}: 2Q depth={d2}, gate counts={dict(isa.count_ops())}")
    print("physical layout (logical qubit i -> physical qubit):", layout)
    return isa, d2, layout

In [16]:
def hardware_stage(ham, meta, results, pairs, n_reps):
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    service = QiskitRuntimeService()
    backend = (service.backend(BACKEND_NAME) if BACKEND_NAME else
               service.least_busy(operational=True, simulator=False, min_num_qubits=2 * meta["norb"]))
    print("backend:", backend.name)
    isa, d2, layout = transpile_for_backend(results[n_reps]["circuit"], backend, meta["norb"], pairs)
    job = SamplerV2(mode=backend).run([isa], shots=SHOTS)
    print("job id:", job.job_id())
    bit_array = job.result()[0].data.meas
    bs, pr = sqd_counts.bit_array_to_arrays(bit_array)
    f_shots, f_unique = report_validity_fractions(bs, pr, meta, f"QPU r{n_reps}")
    sweep = sqd_sweep(ham, meta, bs, pr, label=f"QPU r{n_reps}")
    jid = job.job_id()
    record = dict(backend=backend.name, job_id=jid, n_reps=n_reps, shots=SHOTS, two_qubit_depth=d2,
                  physical_layout=layout, valid_shot_fraction=f_shots, valid_distinct_fraction=f_unique,
                  sqd_sweep={str(k): s for k, s in sweep.items()}, raw_counts=bit_array.get_counts())
    json.dump(record, open(f"{RESULTS_DIR}/hardware_run_{jid}.json", "w"), indent=2, default=str)
    try:
        cal = job.properties()
        if cal is not None:
            json.dump(cal.to_dict(), open(f"{RESULTS_DIR}/calibration_{jid}.json", "w"), indent=2, default=str)
    except Exception as e:
        print("calibration snapshot unavailable:", e)
    print("saved to", os.path.abspath(RESULTS_DIR))
    return record

In [17]:
def circuit_from_saved(n_reps, params_path=None):
    ham, meta, t1, t2 = load_system()
    pairs = local_pairs(meta["norb"])
    prob = Problem(ham, meta)
    x = np.load(params_path or f"{RESULTS_DIR}/lucj_local_r{n_reps}_params.npy")
    qc = verify_on_circuit(prob, x, n_reps, pairs)       # re-checks ordering, sector, energy (seconds)
    return ham, meta, qc, pairs

In [18]:
def submit_job(n_reps, service_name=None, backend_name=None):
    """Transpile + submit, print and SAVE the job id immediately, then return (job, context)."""
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    service = QiskitRuntimeService(name=service_name) if service_name else QiskitRuntimeService()
    backend = service.backend(backend_name or BACKEND_NAME) if (backend_name or BACKEND_NAME) else \
        service.least_busy(operational=True, simulator=False, min_num_qubits=8)
    ham, meta, qc, pairs = circuit_from_saved(n_reps)
    isa, d2, layout = transpile_for_backend(qc, backend, meta["norb"], pairs)
    job = SamplerV2(mode=backend).run([isa], shots=SHOTS)
    jid = job.job_id()
    print("SUBMITTED. job id:", jid, "(keep this; you can fetch results later even if Colab disconnects)")
    json.dump(dict(job_id=jid, backend=backend.name, n_reps=n_reps, shots=SHOTS,
                   two_qubit_depth=d2, physical_layout=layout),
              open(f"{RESULTS_DIR}/submitted_r{n_reps}_{jid}.json", "w"), indent=2)
    return job, dict(ham=ham, meta=meta, d2=d2, layout=layout, backend=backend, n_reps=n_reps)


In [19]:
def process_job(job_or_id, ctx, service_name=None):
    """Wait for / fetch the job, then valid fraction + SQD sweep + save everything."""
    if isinstance(job_or_id, str):
        from qiskit_ibm_runtime import QiskitRuntimeService
        service = QiskitRuntimeService(name=service_name) if service_name else QiskitRuntimeService()
        job = service.job(job_or_id)
    else:
        job = job_or_id
    jid = job.job_id()
    bit_array = job.result()[0].data.meas
    bs, pr = sqd_counts.bit_array_to_arrays(bit_array)
    meta, ham, n_reps = ctx["meta"], ctx["ham"], ctx["n_reps"]
    f_shots, f_unique = report_validity_fractions(bs, pr, meta, f"QPU r{n_reps}")
    json.dump(dict(job_id=jid, raw_counts=bit_array.get_counts()),
              open(f"{RESULTS_DIR}/hardware_counts_{jid}.json", "w"), default=str)   # save raw data BEFORE the slow sweep
    sweep = sqd_sweep(ham, meta, bs, pr, label=f"QPU r{n_reps}")
    record = dict(backend=ctx["backend"].name, job_id=jid, n_reps=n_reps, shots=SHOTS,
                  two_qubit_depth=ctx["d2"], physical_layout=ctx["layout"],
                  valid_shot_fraction=f_shots, valid_distinct_fraction=f_unique,
                  sqd_sweep={str(k): s for k, s in sweep.items()})
    json.dump(record, open(f"{RESULTS_DIR}/hardware_run_{jid}.json", "w"), indent=2, default=str)
    try:
        cal = job.properties()
        if cal is not None:
            json.dump(cal.to_dict(), open(f"{RESULTS_DIR}/calibration_{jid}.json", "w"), indent=2, default=str)
    except Exception as e:
        print("calibration snapshot unavailable:", e)
    print("saved to", os.path.abspath(RESULTS_DIR))
    return record

In [20]:
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token=getpass("IBM Quantum API token: "),
    instance=getpass("CRN: "),
    overwrite=True,
    set_as_default=True,
)

In [22]:
service = QiskitRuntimeService()          # uses the saved account/instance

for b in service.backends():
    st = b.status()
    print(b.name, b.num_qubits, "operational" if st.operational else "down",
          "queue:", st.pending_jobs)

ibm_boston 156 operational queue: 321
ibm_fez 156 operational queue: 28
ibm_kingston 156 operational queue: 22
ibm_marrakesh 156 operational queue: 16
ibm_phoenix 120 operational queue: 5004
ibm_pittsburgh 156 operational queue: 2


In [26]:
job, ctx = submit_job(4)             # prints and saves the job id immediately

   circuit energy error 0.3668 mHa | ffsim-subspace error 0.3668 mHa | weight in (Na,Nb) sector = 1.00000000 | <Na>=2.0000 <Nb>=2.0000
   ffsim LUCJ pass manager: 2Q depth=65, total depth=303
   preset opt-level 3: 2Q depth=96, total depth=466
USING ffsim LUCJ pass manager: 2Q depth=65, gate counts={'sx': 622, 'rz': 548, 'cz': 204, 'measure': 8, 'barrier': 1}
physical layout (logical qubit i -> physical qubit): [59, 58, 57, 56, 69, 68, 67, 66]


/tmp/ipykernel_3773/408067848.py:9: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=backend).run([isa], shots=SHOTS)


SUBMITTED. job id: dbh2419t9uhs73c98u40 (keep this; you can fetch results later even if Colab disconnects)


In [25]:
import glob
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    instance='crn:v1:bluemix:public:quantum-computing:us-east:a/4f8bdaf7d7ab4ed6b9a751430c03446e:ec97513f-3255-47d6-867a-2c6e6fe18ae2::'
)
job = service.job('dbd5hdni0auc7394o400')
print(job.status())            # wait until this says DONE

# Rebuild ham/meta from the repo files (no saved params needed)
ham, meta, t1, t2 = load_system()

# Depth/layout from the file saved at submit time, if it survived
saved = glob.glob(f"{RESULTS_DIR}/submitted_r*_dbd5hdni0auc7394o400.json")
info = json.load(open(saved[0])) if saved else {}

ctx = dict(ham=ham, meta=meta, n_reps=3,              # set to the n_reps you actually submitted
           backend=job.backend(),
           d2=info.get("two_qubit_depth"), layout=info.get("physical_layout"))

record = process_job(job, ctx)

DONE
[QPU r3] valid-shot fraction 0.6945 | valid distinct bitstrings 0.1447 | unique sampled 235
[QPU r3] samples_per_batch= 5: error 41.8418 +/- 5.2876 mHa (n_seeds=10)
[QPU r3] samples_per_batch=10: error 31.9285 +/- 7.4380 mHa (n_seeds=10)
[QPU r3] samples_per_batch=15: error 16.4806 +/- 3.1141 mHa (n_seeds=10)
[QPU r3] samples_per_batch=20: error 4.8465 +/- 1.5653 mHa (n_seeds=10)
[QPU r3] samples_per_batch=25: error 1.4530 +/- 1.1721 mHa (n_seeds=10)
[QPU r3] samples_per_batch=30: error 0.3813 +/- 0.5347 mHa (n_seeds=10)
[QPU r3] samples_per_batch=36: error -0.0000 +/- 0.0000 mHa (n_seeds=10)
saved to /content/SQD/results


In [ ]:
record = process_job(job, ctx)       # waits for the result, then SQD sweep

: 

In [21]:
if __name__ == "__main__":
    ham, meta, prob, pairs, results = noiseless_stage()
    if RUN_HARDWARE:
        for n_reps in N_REPS_LIST:
            hardware_stage(ham, meta, results, pairs, n_reps)

Exact sector energy error: -1.24e-08 mHa | HF error: 53.119 mHa

=== local pairs, n_reps=3 ===
   n_params=85; raw CCSD-seeded error 38.841 mHa
   seed: error 0.3761 mHa (nit=291)
   seed+noise0: error 0.3756 mHa (nit=202)
   seed+noise1: error 0.3837 mHa (nit=270)
   seed+noise2: error 0.3729 mHa (nit=173)
   VARIATIONAL error after minimization: 0.3729 mHa
   circuit energy error 0.3729 mHa | ffsim-subspace error 0.3729 mHa | weight in (Na,Nb) sector = 1.00000000 | <Na>=2.0000 <Nb>=2.0000
[sim r3] valid-shot fraction 1.0000 | valid distinct bitstrings 1.0000 | unique sampled 12
[sim r3] samples_per_batch= 5: error 10.7074 +/- 2.9593 mHa (n_seeds=10)
[sim r3] samples_per_batch=10: error 2.4166 +/- 0.2619 mHa (n_seeds=10)
[sim r3] samples_per_batch=15: error 0.3647 +/- 0.0000 mHa (n_seeds=10)
[sim r3] samples_per_batch=20: error 0.3647 +/- 0.0000 mHa (n_seeds=10)
[sim r3] samples_per_batch=25: error 0.3647 +/- 0.0000 mHa (n_seeds=10)
[sim r3] samples_per_batch=30: error 0.3647 +/- 0.00

/tmp/ipykernel_21972/3273670982.py:8: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=backend).run([isa], shots=SHOTS)


job id: dbcudtpgv2cc738h250g


: 